# Notebook 2: Feature Engineering and Customer Profile Construction
The second notebook will be used as an envrioment for feature extraction and dataset generation. The dataset we will be using to extract features from is the customer-data-rich dataset "Consumer_Meta". These features will be used to enrich the current merged dataset. The aim is to enrich the data with more behavioural features in order to give the models a better chance to identify more pronounced behavioural clusters.

## 0.1 - Imports

In [44]:
# Load and initilise imports:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

## 0.2 - Importing and Loading Datasets

In [45]:
# Load merged_df from preprocessing:
merged_df = pd.read_csv('../0_Data/processed_data/merged_df.csv')

# Load in the third dataset for inspection
customer_meta = pd.read_csv('../0_Data/raw_data/Consumer_meta.csv', skiprows = 1)

# Create copies of the original daatsets preseving the originals
cust_df = customer_meta.copy()
merged = merged_df.copy()

# Initilise a random seed value to be used, ensuring reproducable results. 
np.random.seed(42)

## 1.0 - Dataset Three Feature Extraction
As mentioned this section will focus on inspecting the customer rich dataset for features that will enrich the final customer profiling dataset, that will be used within the machine learning pipeline. First, the dataset will be insepcted to identify features that are currently present.

### 1.1 - Dataset Shape
Below we will display the shape of the dataset to start building an understanding of how many observations and features there is present within the data.

In [46]:
# Print the shape of dataset 3:
print("Dataset Three Shape:", cust_df.shape)

Dataset Three Shape: (20000, 20)


### 1.2 - Health Summary
A health summary will be displayed as a follow-on from the shape inspection. This is done to provide deeper context and understanding of the behaviour of the data we are working with. 

In [47]:
# Initlise the function: 
def health_summary(df):
    report = pd.DataFrame({
        'Data Type  ': df.dtypes,
        'Null Values  ': df.isnull().sum(),
        'Unique Values  ': df.nunique(),
        'Duplicate Values   ': df.duplicated().sum()
    })
    return report

# Print the health summaries for both dataframes:
print("-----------     Dataset Three Health Summary     ------------")
display(health_summary(cust_df))

-----------     Dataset Three Health Summary     ------------


,Data Type,Null Values,Unique Values,Duplicate Values
transaction_id,int64,0,20000,0
timestamp,str,0,19992,0
store_id,int64,0,45,0
city,str,0,9,0
country,str,0,4,0
store_type,str,0,3,0
product_category,str,0,6,0
product_name,str,0,43,0
unit_price,float64,0,359,0
quantity,int64,0,9,0


The health summary displays an extensive list of features (20) with multple peices of inportant information. The most important is the amount of null values. These will need to be preprocessed before extracting the features. However, it makes more sense to identify the features that will be extracted before preprocessing, as to avoid any work being carried out on features that are going to be removed.

## 2.0 - Identifying Valuable Features
This section will revolve around the process of identifying which features are suitable to be extracted and will be benificial in extending the comprehension of our current data. Determining which features are valuable and which are not will come down to two aspects. The first is:

- Is the data contained within this feature currently present within the preprocessed merged dataset?

 and if not, the second aspect is:

- critically analysising whether or not the data contained within the current feature will add any value to the merged dataset?

### 2.1 - Feature List
To start, we will print a list of all the features currently contained within the dataset. 

In [48]:
# Implement a function that prints features and values:
def table_inspection(df):
    for col in df.columns:
        print(col, " - ","Example Value:", df[col].iloc[0])
# Call the above function:
print(table_inspection(cust_df))

transaction_id  -  Example Value: 10001
timestamp  -  Example Value: 2023-01-01 00:39:39
store_id  -  Example Value: 35
city  -  Example Value: Melbourne
country  -  Example Value: AUS
store_type  -  Example Value: Mall Kiosk
product_category  -  Example Value: Coffee
product_name  -  Example Value: Double Espresso
unit_price  -  Example Value: 3.04
quantity  -  Example Value: 1
discount_applied  -  Example Value: True
payment_method  -  Example Value: Credit Card
customer_id  -  Example Value: qiyfrwsk
customer_age_group  -  Example Value: 35-44
customer_gender  -  Example Value: Female
loyalty_member  -  Example Value: False
weather_condition  -  Example Value: Sunny
temperature_c  -  Example Value: 22.2
holiday_name  -  Example Value: New Year's Day
total_amount  -  Example Value: 2.74
None


Using the table above to identify valuable features we can come to a conclusion. The only features that pass the criteria previously set out are:

- Loyalty_member
- discount_applied

All other features can either be considered repeated data for what is already present within merged_df or features that do not enrich the data further, within the context of consumer behaviour. With the features identified we can drop all other non relevant ones and begin preprocessing the remaining ones.  

In [49]:
# Create a variable that stores each of the pre determined features going to be dropped:
drop_features = ['transaction_id',
                'timestamp',
                'country', 
                'customer_id',
                'weather_condition',
                'temperature_c',
                'holiday_name', 
                'store_id', 
                'city', 
                'store_type', 
                'product_category', 
                'product_name', 
                'unit_price', 
                'quantity',
                'customer_age_group', 
                'customer_gender', 
                'total_amount', 
                'payment_method']

# Drop the columns
cust_df = cust_df.drop(columns=drop_features)

# Confirm feature removal was successful:
print("Remaining Features:")
print("")
print(cust_df.columns)

Remaining Features:

Index(['discount_applied', 'loyalty_member'], dtype='str')


## 3.0 - Feature Preprocessing
From the health summary generated before we already have some information about our remaining features. Firstly there are no null values or duplicates seen within any of the observations. The issue we do have is concerned with data types. Similarly to the preprocessing completed for the first two datasets, we need to convert any non-numerical features to numerical ones. Both discount Applied and Loyalty member are boolean's which means they are already binary and map directly to a numerical system.

### 3.1 - discount_applied & loyalty_member Binary Conversion

In [50]:
# Convert discount_applied and loyalty_member to numerical values: 
# discount_applied [YES = 1, NO = 0]:
cust_df['discount_applied'] = cust_df['discount_applied'].astype(int)
# loyalty_member [YES = 1, NO = 0]:
cust_df['loyalty_member'] = cust_df['loyalty_member'].astype(int)
# Confirmation of conversion:
# Initilise a function that produces a list of values for a given feature:
def print_values(df, feature):
    print("Unique Count:", df[feature].nunique())
    print("Value:", df[feature].unique())

# Call the function for each feature: 
print("")
print("---- discount_applied values ----")
print_values(cust_df, 'discount_applied')
print("")
print("---- loyalty_member values ----")
print_values(cust_df, 'loyalty_member')
    



---- discount_applied values ----
Unique Count: 2
Value: [1 0]

---- loyalty_member values ----
Unique Count: 2
Value: [0 1]


Both, discount_applied and loyalty_member's values have been successfuly converted to numercial datatypes (Binary). 

In [51]:
def feature_title_list(df):
    for feature in df.columns: 
        print(feature)

# Confirm new features: 
feature_title_list(cust_df)

discount_applied
loyalty_member


## 4.0 Building customer_features
customer_features is the final step of all the previous preprocessing work carried out before this point. customer_features is a new dataset that is built by combining the features from merged_df and the third dataset we have been working on in this notebook. The features in customer_features will be built by grouping merged_df by customer_id in order to generate a dataset where each row correlates to one customer. Based on the features we have been working with, customer_features will include: 

- total_spend
- average_transaction_value
- visit_frequency
- item_count
- cancellation_rate
- unique_products
- time_of_day
- time_of_week
- discount_applied
- loyalty_member

similarly to the previous preprocessing steps, we will work down the list in the order that it appears.

### 4.1 - Dataset Initilisation
In order to maintain strucutre within the notebook, the code block below will be dedicated to initlising the empty dataset 'customer_features' for all subsequent code blocks to add to the dataset as they are executed. 

In [52]:
# Initlise empty dataset
customer_features = pd.DataFrame()

### 4.2 - total_spend

total_spend represents the total amount of money the customer has spent in the cafe within the last 2 years. This featues is calculated in two stages. The first is line_total. This feature is created for every obseveration and is calculated by multipling the quantity and price of the products bought in that specific invoice. Following this, line_total is grouped by customer_id and a sum of every line total value associated per customer is calculated. reset_index() is used here to revert customer_id from being implemented as a row index. If left out, customer_id would not be a normal column but rather an index down the side of the dataset. 

In [53]:
# Calculate line_total for each observation: 
merged['line_total'] = merged['quantity'] * merged['price']

# Group all line_total values together and sum them
customer_features = merged.groupby('customer_id')['line_total'].sum().reset_index()

# rename feature titles to ensure they are clear: 
customer_features.columns = ['customer_id', 'total_spend']

# Display an exaple of how total_spend works:
customer_features.head()

,customer_id,total_spend
0,12346,258.68
1,12347,4921.53
2,12348,1817.80
3,12349,4404.54
4,12350,334.40


### 4.3 - visit_frequency
visit_frequencty represents the total number of visits the customer made over the recorded time period. It is calculated by summing up all of the purchases each individual customer made. It is important to note however, when performing the preprocessing steps for merged_df it was identified that invoices are split by products bought not by unique invoive number. Meaning a purchase made on the same day is spread accross multiple observtions. This means that calculating total visits cannot be done by summing the total number of observations per customer, rather how many unique invoive ids there are per customer. 

In [54]:
# Calculate total visits - grouped by unique invoices per customer_id
total_visits = merged.groupby('customer_id')['invoice'].nunique().reset_index()

# Rename columns:
total_visits.columns = ['customer_id', 'total_visits']

# Add total visits to customer_features 
customer_features = customer_features.merge(total_visits, on='customer_id')

# Display the addition to customer_features: 
customer_features.head()

,customer_id,total_spend,total_visits
0,12346,258.68,17
1,12347,4921.53,8
2,12348,1817.80,5
3,12349,4404.54,5
4,12350,334.40,1


### 4.4 - average_transaction_value
average_transaction_value is a popular metric used in businesses to determine the average spend a user makes over the course of all of their visits. It is paired with total_spend however provides slightly better statistical information. It is calculated by dividing total_spend by number of transactions for each customer. 

In [55]:
# Calculate ATV by dividing total spend by visits: 
customer_features['atv'] = customer_features['total_spend'] / customer_features['total_visits']

# Call a head() function to display changes:
customer_features.head()

,customer_id,total_spend,total_visits,atv
0,12346,258.68,17,15.216471
1,12347,4921.53,8,615.191250
2,12348,1817.80,5,363.560000
3,12349,4404.54,5,880.908000
4,12350,334.40,1,334.400000


### 4.5 - total_item_count
total_item_count represents the total number of products each customer hs bough across all transactions. 

In [56]:
# initilise total_item_count and grouby by customer_id
total_item_count = merged.groupby('customer_id')['quantity'].sum().reset_index()

# Rename columns:
total_item_count.columns = ['customer_id', 'total_item_count']

# Add the feature to the dataset:
customer_features = customer_features.merge(total_item_count, on='customer_id')

# Print the updated head():
customer_features.head()

,customer_id,total_spend,total_visits,atv,total_item_count
0,12346,258.68,17,15.216471,66
1,12347,4921.53,8,615.191250,2967
2,12348,1817.80,5,363.560000,2234
3,12349,4404.54,5,880.908000,1619
4,12350,334.40,1,334.400000,197


### 4.6 - cancellation_rate
cancellation_rate is a percentage value that represents the amount of transaction a customer cancels or fails. Customers who have not cancelled any transaction are artifically populated with a 0 value to avoid null or NaN values being present. 

In [57]:
# Sum unique cancancelled transactions per customer
cancelled_invoices = merged[merged['order_cancelled'] == True].groupby('customer_id')['invoice'].nunique()

# Initilise feature using percentage of cancelled transactions as the value;
cancellation_rate = (cancelled_invoices / customer_features.set_index('customer_id')['total_visits']).reset_index()

# Rename feature:
cancellation_rate.columns = ['customer_id', 'cancellation_rate']

# Add feature to customer_features:
customer_features = customer_features.merge(cancellation_rate, on='customer_id')

# Fill customers who have 0% cancellation rate with 0:
customer_features['cancellation_rate'] = customer_features['cancellation_rate'].fillna(0) 

# Display the feature addition to the dataset:
customer_features.head()


,customer_id,total_spend,total_visits,atv,total_item_count,cancellation_rate
0,12346,258.68,17,15.216471,66,0.294118
1,12347,4921.53,8,615.191250,2967,0.000000
2,12348,1817.80,5,363.560000,2234,0.000000
3,12349,4404.54,5,880.908000,1619,0.200000
4,12350,334.40,1,334.400000,197,0.000000


### 4.7 - unique_products
unique_products represent the number of individual products each customer purchases across all transactions.

In [58]:
# Count the number of unique products each customer purchases:
unique_products = merged.groupby('customer_id')['stockcode'].nunique().reset_index()

# Rename the feature:
unique_products.columns = ['customer_id', 'unique_products']

# Add the feature to customer_features:
customer_features = customer_features.merge(unique_products, on = 'customer_id')

# Display the upated dataset head()
customer_features.head()

,customer_id,total_spend,total_visits,atv,total_item_count,cancellation_rate,unique_products
0,12346,258.68,17,15.216471,66,0.294118,30
1,12347,4921.53,8,615.191250,2967,0.000000,126
2,12348,1817.80,5,363.560000,2234,0.000000,25
3,12349,4404.54,5,880.908000,1619,0.200000,139
4,12350,334.40,1,334.400000,197,0.000000,17


### 4.8 - time_of_day
time_of_day is an integer value assigned to each customer, describing the most frequent time of day a particular customer tends to shop - represented on a 24 hour clock (zero indexed: 0-23).

In [59]:
# Initilise variable and populate with each customer's most frequent shopping hour.
time_of_day = merged.groupby('customer_id')['hour'].agg(lambda x: x.mode()[0]).reset_index()

# Rename feature:
time_of_day.columns = ['customer_id', 'time_of_day']

# Add the feature to customer_features
customer_features = customer_features.merge(time_of_day, on='customer_id')

# Print the updated head():
customer_features.head()

,customer_id,total_spend,total_visits,atv,total_item_count,cancellation_rate,unique_products,time_of_day
0,12346,258.68,17,15.216471,66,0.294118,30,13
1,12347,4921.53,8,615.191250,2967,0.000000,126,14
2,12348,1817.80,5,363.560000,2234,0.000000,25,14
3,12349,4404.54,5,880.908000,1619,0.200000,139,9
4,12350,334.40,1,334.400000,197,0.000000,17,16


### 4.9 - time_of_week
Similar to time_of_day this feature represents the most frequent day of the week each customer tends to shop, also represented as a zero indexed integer. 

In [60]:
# Initilise feature and calculate the most frequent day of the week each customer shops:
day_of_week = merged.groupby('customer_id')['day_of_week'].agg(lambda x: x.mode()[0]).reset_index()

# Rename Feature:
day_of_week.columns = ['customer_id', 'day_of_week']

# Add feature to customer_features dataset:
customer_features = customer_features.merge(day_of_week, on='customer_id')

# Print the updated dataset head()
customer_features.head()

,customer_id,total_spend,total_visits,atv,total_item_count,cancellation_rate,unique_products,time_of_day,day_of_week
0,12346,258.68,17,15.216471,66,0.294118,30,13,0
1,12347,4921.53,8,615.191250,2967,0.000000,126,14,1
2,12348,1817.80,5,363.560000,2234,0.000000,25,14,0
3,12349,4404.54,5,880.908000,1619,0.200000,139,9,3
4,12350,334.40,1,334.400000,197,0.000000,17,16,2


### 4.0 - discount_rate
discount_rate represents how often a cusomter tends to shop with a discount. The values for this feature are synthetically created using the distribution of discounts found within the third dataset. Each transaction is then randomly assigned a discount value in accordance to the distribution rate pulled from the third dataset. 

In [61]:
# Calculate the distribution of discounts used from Dataset 3. 
discount_distribution = cust_df['discount_applied'].value_counts(normalize=True)

# Apply the discount distribution rate to synthetically populate customer_features:
merged['discount_applied'] = np.random.choice(
    discount_distribution.index,
    size=len(merged),
    p=discount_distribution.values
)

# Calcualte each customers average discount rate:
discount_rate = merged.groupby('customer_id')['discount_applied'].mean().reset_index()

# Rename Colunmns:
discount_rate.columns = ['customer_id', 'discount_rate']

# Add feature to customer_features dataset:
customer_features = customer_features.merge(discount_rate, on='customer_id')

# Display the updated dataset head()
customer_features.head()

,customer_id,total_spend,total_visits,atv,total_item_count,cancellation_rate,unique_products,time_of_day,day_of_week,discount_rate
0,12346,258.68,17,15.216471,66,0.294118,30,13,0,0.153846
1,12347,4921.53,8,615.191250,2967,0.000000,126,14,1,0.121622
2,12348,1817.80,5,363.560000,2234,0.000000,25,14,0,0.085106
3,12349,4404.54,5,880.908000,1619,0.200000,139,9,3,0.105556
4,12350,334.40,1,334.400000,197,0.000000,17,16,2,0.117647


### 4.0 - loyalty_member
This feature required more purposeful initiative before implementing. Loyalty memeber represents whether or not a customer is a loyalty memeber or not. 

In [62]:
# Calculate the distribution of loyalty members in dataset 3:
loyalty_distribution = cust_df['loyalty_member'].value_counts(normalize=True)

# Pull each unique customer and assign them a loyalty value (0, 1) using the same distribution:
unique_customers = merged['customer_id'].unique()
# initlise a small dataframe to store customer_id and the newly assigned loyalty member status:
loyalty_lookup = pd.DataFrame({
    'customer_id': unique_customers,
    'loyalty_member': np.random.choice(
        loyalty_distribution.index,
        size = len(unique_customers),
        p = loyalty_distribution.values
    )
})

# Add the feature to the customer_features dataset:
customer_features = customer_features.merge(loyalty_lookup, on= 'customer_id')

# Display the updated dataset head():
customer_features.head()

,customer_id,total_spend,total_visits,atv,total_item_count,cancellation_rate,unique_products,time_of_day,day_of_week,discount_rate,loyalty_member
0,12346,258.68,17,15.216471,66,0.294118,30,13,0,0.153846,1
1,12347,4921.53,8,615.191250,2967,0.000000,126,14,1,0.121622,0
2,12348,1817.80,5,363.560000,2234,0.000000,25,14,0,0.085106,0
3,12349,4404.54,5,880.908000,1619,0.200000,139,9,3,0.105556,0
4,12350,334.40,1,334.400000,197,0.000000,17,16,2,0.117647,0


## 5.0 - customer_features Dataset Inspection 

Now the features have been created for customer_features an inspection on the new dataset is to be carried out starting with dataset shape. 

### 5.1 customer_features shape

In [63]:
# Print the number of observations and number of features in the dataset: 
customer_features.shape

(5942, 11)

The results from the dataset shape code confirms that there are 5942 unique customers that have values across 11 different features. 

### 5.2 - customer_features Health Summary:
Similarly to the preprocessing steps take in the previous notebook, a health summary will be carried out on the new dataset. This acts as a check for any issues that may be present within any of the observations. 

In [64]:
# Print a health summary for customer_features
health_summary(customer_features)

,Data Type,Null Values,Unique Values,Duplicate Values
customer_id,int64,0,5942,0
total_spend,float64,0,5836,0
total_visits,int64,0,101,0
atv,float64,0,5829,0
total_item_count,int64,0,2375,0
cancellation_rate,float64,0,254,0
unique_products,int64,0,467,0
time_of_day,int64,0,14,0
day_of_week,int64,0,7,0
discount_rate,float64,0,1609,0


The health summary above confirms that the dataset 'customer_features' contains all of the features created; with the correct naming conventions as well as all datatypes are numeric and there are no null or duplicate values present. With the health summary providing successful results, the dataset can be converted to csv and have StandardScalar applied. 

## 6.0 StandardScalar 
In this final section, the customer_features dataset will be converted to a csv file. This is followed by the creation of another dataset called 'scaled_features'. This dataset will represent the scaled version of the customer_features dataset. 

The reason there will be two versions of customer_features created is so the unscaled version will be where real, readable values are preseverd to be used during the analysis phase and a scaled version to be used to feed into clustering Models. 

In [65]:
# Save the unscaled version to csv: 
customer_features.to_csv('customer_features.csv', index=False)

# Initilise list of features that will be scaled
feature_cols = ['total_spend', 'atv', 'total_visits', 'total_item_count',
       'cancellation_rate', 'unique_products', 'time_of_day', 'day_of_week',
       'discount_rate', 'loyalty_member']

# Initilise StandardScalar: 
scaler = StandardScaler()

# Implement scaled dataset using StandardScalar: 
scaled_features = scaler.fit_transform(customer_features[feature_cols])
scaled_df = pd.DataFrame(scaled_features, columns=feature_cols)
scaled_df['customer_id'] = customer_features['customer_id']
scaled_df.to_csv('customer_features_scaled.csv', index=False)
